# Extent callback WIP developer notebook

This notebook demonstrates `GISDocument.bind_extent_slice_callback` with a STAC-backed `xarray.DataArray`.

Use it to verify:
- extent -> bbox conversion,
- viewport slicing/resampling,
- optional index processing,
- optional tiler layer creation from computed output.

This is a developer-facing workflow to get feedback on the API surface and such (not end-user docs).

In [ ]:
from typing import Iterable, List

from jupytergis.tiler import GISDocument
from pyproj import Transformer
import pystac_client
import stackstac


def extent_3857_to_bbox_4326(extent: Iterable[float]) -> List[float]:
    """
    Convert Web Mercator extent to STAC-ready lon/lat bbox.

    Input:  [minx, miny, maxx, maxy] in EPSG:3857
    Output: [min_lon, min_lat, max_lon, max_lat] in EPSG:4326
    """
    minx, miny, maxx, maxy = extent
    transformer = Transformer.from_crs(
        "EPSG:3857",
        "EPSG:4326",
        always_xy=True,
    )
    min_lon, min_lat = transformer.transform(minx, miny)
    max_lon, max_lat = transformer.transform(maxx, maxy)
    return [min_lon, min_lat, max_lon, max_lat]


doc = GISDocument("footprints.jGIS")

In [ ]:
# `doc.extent` is in map CRS (default EPSG:3857); convert for STAC queries.
m = doc.extent
bbox_doc = extent_3857_to_bbox_4326([m.west, m.south, m.east, m.north]) if m else None

# Limit the AoI when we fetch the collection
bbox_search = [
    -18.248776440117897,
    27.004514659408795,
    -16.033094271404384,
    34.01156206721229,
]

bbox = bbox_search
# bbox_doc, bbox

In [ ]:
# URL = "https://stac.dataspace.copernicus.eu/v1"
URL = "https://earth-search.aws.element84.com/v1"
catalog = pystac_client.Client.open(URL)

In [ ]:
items = catalog.search(
    bbox=bbox,
    collections=["sentinel-2-l2a"],
    datetime="2020-03-01/2020-03-02",
).item_collection()
len(items)

In [ ]:
# Optional: visualize STAC item footprints as a vector layer.
# `items` is a PySTAC ItemCollection

# doc.add_geojson_layer(data=items.to_dict(), name="stac-footprints")

In [ ]:
# Create the DataArray from the STAC items
da = stackstac.stack(items, epsg=4326, bounds_latlon=bbox)

## Register extent callback

This callback runs whenever `options.extent` changes (pan/zoom).

notes:
- `on_slice` receives the sliced array (lazy if `compute=False`, concrete if `compute=True`)
- `process` is applied to each slice before `on_slice`.
- `add_tiler_layer_on_compute=True` pushes computed slices to jupyter-tiler to create a layer

In [ ]:
# State object to hold lazy and computer arrays
# The computed value is only used if `compute` is False when binding the callback below
# `Lazy` holds the non-computed slice
state = {"lazy": None, "computed": None}


# Define functions to process sliced arrays
def ndvi_process(sliced):
    red = sliced.sel(band="red")
    nir = sliced.sel(band="nir")
    ndvi = (nir - red) / (nir + red)
    ndvi = ndvi.where((nir + red) != 0)
    ndvi.name = "ndvi"
    # Collapse time for a single 2D layer preview.
    if "time" in sliced.dims:
        return ndvi.mean(dim="time", skipna=True, keep_attrs=True)
    return ndvi


def ndwi_process(sliced):
    green = sliced.sel(band="green")
    nir = sliced.sel(band="nir")
    ndwi = (green - nir) / (green + nir)
    ndwi = ndwi.where((green + nir) != 0)
    ndwi.name = "ndwi"
    # Collapse time for a single 2D layer preview.
    if "time" in sliced.dims:
        return ndwi.mean(dim="time", skipna=True, keep_attrs=True)
    return ndwi


sub_id = doc.bind_extent_slice_callback(
    da,
    on_slice=lambda out: state.update({"lazy": out}),
    data_crs="EPSG:4326",
    x_name="x",
    y_name="y",
    process=ndwi_process,
    compute=False,
    strict_process=False,
    viewport_width=640,
    viewport_height=360,
    resample_method="linear",
    add_tiler_layer_on_compute=False,
    tiler_layer_kwargs={
        "name": "NDWI viewport",
        "colormap_name": "viridis",
        "rescale": (-1.0, 1.0),
        "opacity": 0.9,
    },
)

In [ ]:
doc

## Interactive step

Display the map widget, then pan/zoom to trigger extent updates.

When done testing, unbind the callback to stop receiving updates.

In [ ]:
doc.unbind_extent_slice_callback(sub_id)

In [ ]:
# Sanity check to make sure this populated, can be empty if the sliced area has no data
state["lazy"]

In [ ]:
import dask.diagnostics

lazy = state["lazy"]
assert lazy is not None, "Pan/zoom the map (or set extent) so the callback runs."
with dask.diagnostics.ProgressBar():
    state["computed"] = lazy.compute()

In [ ]:
await doc.add_tiler_layer(
    state["computed"],
    name="ndwi-manual-compute",
    colormap_name="plasma",
    rescale=(-1, 1),
    reproject="max",
)